# Augmented Lagrangian Blueprint

# Einfaches Beispiel: 2D Structured Mesh mit 1D Gauss-Verteilungen


In [76]:
#print(help(H1))
#print("\n\n")
#print(help(HDiv))
#print("\n\n")
#print(help(FESpace))
#print(help(comp.ComponentGridFunction.Set))
print(help(MakeStructured2DMesh().Boundaries))

Help on method Boundaries in module ngsolve.comp:

Boundaries(...) method of ngsolve.comp.Mesh instance
    Boundaries(*args, **kwargs)
    Overloaded function.

    1. Boundaries(self: ngsolve.comp.Mesh, pattern: str) -> ngsolve.comp.Region

    Return boundary mesh-region matching the given regex pattern

    2. Boundaries(self: ngsolve.comp.Mesh, bnds: collections.abc.Sequence[typing.SupportsInt | typing.SupportsIndex]) -> ngsolve.comp.Region

    Generate boundary mesh-region by boundary condition numbers

None


## Variablen und Mesh + Hauptschleife

## Update in $\varphi$, löse PDE

In [230]:
def update_phi(mu, phi, q, rho0, rho1, mesh, r=1, epsilon=1e-6):
    # Wie im Paper die perturbierte Variante wählen
    
    rho, m = mu.components
    a, b   = q.components

    V = phi.space

    phi_new = GridFunction(V)

    u = V.TrialFunction() # TODO umbenennen um Verwirrung zu vermeiden
    v = V.TestFunction()

    lhs = BilinearForm(V)
    rhs = LinearForm(V)

    lhs += r * grad(u) * grad(v) * dx
    lhs += r * epsilon * u * v * dx
    #lhs += r*(-grad(phi)*grad(v)+epsilon*phi*v)*dx

    rhs += (
         (rho - r*a) * grad(v)[1]
        - (m - r*b) * grad(v)[0]
        
    ) * dx
    #rhs += 
    rhs += (-1)*rho0*v.Trace()*ds(definedon=mesh.Boundaries("bottom")) # implements initial condition
    rhs += 1*rho1*v.Trace()*ds(definedon=mesh.Boundaries("top")) # implements final 
    #rhs += (mu - r*div(q)) * v * dx 

    lhs.Assemble()
    rhs.Assemble()
    #print(lhs.mat)
    phi_new.vec.data = lhs.mat.Inverse(V.FreeDofs()) * rhs.vec
    #print(phi_new.vec.data)
    return phi_new

def update_gradient(phi, W):
    grad_phi = GridFunction(W*W)
    grad_phi.components[0].Set(grad(phi)[0])
    grad_phi.components[1].Set(grad(phi)[1])
    #print(grad_phi)
    return grad_phi

## Update in $q$ mit Newton-Schritt

In [251]:
# Wo gehts schief?
import numpy as np


def newton(alpha, beta, tol=1e-10, maxit=50):

    # Wenn bereits in der Menge K, einfach zurückgeben
    if alpha + 0.5 * np.dot(beta, beta) <= 0:

        return alpha, beta

    # Wenn nicht: Newton-Iteration
    beta2 = np.dot(beta, beta)

    lam = 0.0 # Lagrange-Multiplikator
    #i = 0
    for k in range(maxit):

        denom = 2.0 + lam

        f = (alpha
            - 0.5 * lam
            + 2.0 * beta2 / (denom * denom))

        # Ableitung
        fp = (
            -0.5
            - 4.0 * beta2 / (denom**3)
        )

        # Definition Newton-Iteration
        lam_new = lam - f / fp
        
        if abs(lam_new - lam) < tol:
            #print(abs(lam_new - lam))
            lam = lam_new
            break

        lam = lam_new
        if k == maxit-1:
            print("Abbruch")

    # In Formeln einsetzen
    a = alpha - 0.5 * lam

    b = 2.0 * beta / (2.0 + lam)

    return a, b


def update_q(mu, phi, q, W, r=1.0):

    rho, m = mu.components
    a, b   = q.components

    # Definition von p := grad(phi) + mu/r
    grad_phi = update_gradient(phi, W)#GridFunction((0,0))#grad(phi)

    alpha = grad_phi.components[1] + rho / r

    beta_cf = m / r + grad_phi.components[0] # FIXME? Sollte jetzt passen?

    # Werte auf den Gitterpunkten
    avec = a.vec.FV().NumPy()
    bvec = b.vec.FV().NumPy()
    rhovec = rho.vec.FV().NumPy()
    #print(rhovec)
    mvec = m.vec.FV().NumPy()

    # Punktweise Update TODO?
    ndof_scalar = len(avec)
    dim = mesh.dim

    for i in range(ndof_scalar):

        alpha = rhovec[i] / r

        #beta = np.array([
        #    mvec[dim * i + j] / r
        #    for j in range(dim)
        #])
        beta = mvec[i] / r

        anew, bnew = newton(alpha, beta)
        
        avec[i] = anew
        bvec[i] = bnew
        #for j in range(dim):
            #bvec[dim * i + j] = bnew[j]
            
    return q

## Update in $\mu$

In [254]:
# Das sollte passen
def update_mu(mu, phi, q, W, r=1.0):
    rho, m = mu.components
    a, b   = q.components

    # grad(phi)[0] = phi_x
    # grad(phi)[1] = phi_t, wenn y die Zeitrichtung ist; Ändern für 2/n-Dimensionalen Fall, dann wäre es grad(phi)[2]
    grad_phi = update_gradient(phi, W)
    phi_x = grad_phi.components[0]
    phi_t = grad_phi.components[1]

    #rho.vec.data += r*(phi_t.vec - a.vec)
    #m.vec.data += r * (phi_x.vec - b.vec)
    rho.Set(rho + r * (phi_t - a))
    m.Set(m + r * (phi_x - b))
    mu.components[0].Set(rho)
    mu.components[1].Set(m)
    return mu

In [255]:
from ngsolve import *
from ngsolve.webgui import Draw
from ngsolve.comp import IntegrationRuleSpace
#import matplotlib.pyplot as plt
#import ipdb
#from OTmeshing import OTMesh
from ngsolve.meshes import MakeStructured2DMesh

# TODO Boundary Conditions korrekt?
# TODO Zieldichte angeben?
# TODO Debugging - Output nicht korrekt

mesh = MakeStructured2DMesh(
    quads=True,
    nx=8,
    ny=8,
    mapping=lambda x, y: (x, y)
)


order = 2

# laut Docs: scalar space, continuous piecewise polynomials
# used for rho (density), a, phi (lagrange multiplier of weak formulation)
V = H1(mesh, order=order)#Periodic(H1(mesh, order=order, dirichlet="bottom|top")) 
V.FreeDofs()[0] = False
V.FreeDofs(True)[0] = False

# vector space
# https://hpfem.org/wp-content/uploads/doc-web/doc-tutorial/src/hermes2d/A-linear/02-space.html
# used for m (momentum / rho * |v|) , b
# In OT-file L2/IntegrationRuleSpace is used?
#W = HDiv(mesh, order=order)
W = IntegrationRuleSpace(mesh, order=order)#Periodic(L2(mesh, order=order))

# Product FEM-spaces
F_mu = W*W
F_q  = W*W

# Lagrange multiplier FEM-space
F_phi = V

mu = GridFunction(F_mu)
rho, m = mu.components
q = GridFunction(F_q)
a, b = q.components
phi = GridFunction(F_phi)

sigma = 0.10


u0 = exp(-((x - 0.25)**2) / (2 * sigma**2))
u1 = exp(-((x - 0.75)**2) / (2 * sigma**2))
u0 = u0 / Integrate(u0, mesh)
u1 = u1 / Integrate(u1, mesh)
ut = (1-y)*u0 + y*u1
ut /= Integrate(ut, mesh)

#rho.Set(u0) #Bringt aber nichts für die Zielverteilung?
rho.Set(ut)
#Draw(rho, mesh)
#rho.Set(u0, definedon=mesh.Boundaries("bottom")) 
#rho.Set(u1, definedon=mesh.Boundaries("top")) # Wie konkret angeben?  

#m.Set(CoefficientFunction((0, 0)))
m.Set(0) # In 1D

a.Set(0)
#b.Set(CoefficientFunction((0, 0)))
b.Set(0) # In 1D

phi.Set(0)

#print(mesh.Boundaries)
#print(type(rho))
#phi = update_phi(mu, phi,q)
#q = update_q(mu,phi,q)
#mu =update_mu(mu, phi,q)

for i in range(5):
    #Draw(phi, mesh)
    phi = update_phi(mu, phi, q, u0, u1, mesh)
    #Draw(phi, mesh)
    #Draw(grad(phi)[1], mesh)
    q = update_q(mu, phi, q, W)
    mu = update_mu(mu, phi, q, W)
    Draw(rho, mesh)    



WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…